# AMTTP Fraud Detection: GPU-Optimized VAE + GNN + Ensemble Pipeline

This notebook implements the definitive approach with a VAE-first architecture, parallel supervised models (GATv2, XGBoost, LightGBM), meta-ensemble stacking, and explainability, optimized for Google Colab GPU with memory safety.


# 1) Environment Setup (GPU, Memory Efficient DataLoaders)

- Ensure GPU (T4/A100) is active in Colab runtime.
- Install optimized packages (PyTorch + PyG + xformers + lightgbm + xgboost + optuna + shap + captum).
- Configure deterministic seeds, mixed precision (`torch.cuda.amp`), gradient checkpointing, and memory-aware loaders.
- Utilities for VRAM-safe batching (dynamic batch sizing, gradient accumulation, neighbor sampling).


In [ ]:
# ====== OPTIMIZED STACK FOR COLAB (Dec 2025) ======
# Colab already has PyTorch + CUDA pre-installed - DON'T upgrade it!

# 1. PyTorch Geometric (auto-detects Colab's torch/CUDA version)
!pip install -q torch-geometric

# 2. PyG sparse/scatter extensions (match Colab's PyTorch)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu$(python -c "import torch; print(torch.version.cuda.replace('.',''))").html

# 3. Speed libs (fast install)
!pip install -q polars numba

# 4. ML utilities
!pip install -q xgboost lightgbm shap

# 5. xformers (optional, for attention optimization - skip if issues)
# !pip install -q xformers

print("✅ Installation complete - using Colab's native PyTorch!")

# ====== IMPORTS & OPTIMIZATIONS ======
import os, random, math, gc
import numpy as np
import pandas as pd
import polars as pl
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import GradScaler, autocast
from numba import jit, prange

# RAPIDS check
try:
    import cudf
    import cuml
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
    from cuml.model_selection import train_test_split as cu_train_test_split
    RAPIDS_AVAILABLE = True
    print('✅ RAPIDS cuML available!')
except ImportError:
    RAPIDS_AVAILABLE = False
    print('ℹ️ RAPIDS cuML not available')

# Optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('highest')

TORCH_COMPILE_AVAILABLE = hasattr(torch, 'compile')
print('✅ torch.compile available' if TORCH_COMPILE_AVAILABLE else 'ℹ️ torch.compile not available')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Your Numba JIT functions (keep them!)
@jit(nopython=True, parallel=True, cache=True)
def fast_threshold_classify(probs, t_critical, t_high, t_medium, t_low):
    n = len(probs)
    result = np.empty(n, dtype=np.int32)
    for i in prange(n):
        p = probs[i]
        if p >= t_critical: result[i] = 4
        elif p >= t_high: result[i] = 3
        elif p >= t_medium: result[i] = 2
        elif p >= t_low: result[i] = 1
        else: result[i] = 0
    return result

# ... (add your other JITs, EMA, etc.)

# ====== EMA (Exponential Moving Average) ======
class EMA:
    """Exponential Moving Average for model weights - improves generalization"""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}
        self.backup = {}
    
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.detach()
    
    def apply_shadow(self, model):
        self.backup = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)
    
    def restore(self, model):
        model.load_state_dict(self.backup)

# ====== MC Dropout for Uncertainty ======
def mc_dropout_predict(model, x, edge_index, n_samples=30):
    """Monte Carlo Dropout for uncertainty estimation"""
    model.train()  # Enable dropout
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            logits = model(x, edge_index)
            preds.append(torch.sigmoid(logits))
    preds = torch.stack(preds)
    mean = preds.mean(dim=0)
    std = preds.std(dim=0)
    model.eval()
    return mean, std

# ====== Memory helper ======
def free_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compile_model(model, mode="max-autotune"):
    if TORCH_COMPILE_AVAILABLE:
        try:
            return torch.compile(model, mode=mode, fullgraph=True)
        except: return model
    return model

print('\n✅ Ultra-fast stack ready!')

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 21.4/23.7 MB 209.2 MB/s eta 0:00:01  Downloading https://pypi.nvidia.com/nvidia-cuda-nvrtc-cu12/nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl (23.7 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 191.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 191.4 MB/s eta 0:00:00
     ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/393.1 MB 210.3 MB/s eta 0:00:02  Downloading https://pypi.nvidia.com/nvidia-cublas-cu12/nvidia_cublas_cu12-12.6.4.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (393.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 91.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 91.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.4/200.2 MB 206.5 MB/s eta 0:00:01  Downloading https://pypi.nvidia.com/nvidia-cufft-cu12/nvid

RuntimeError: cannot cache function 'fast_threshold_classify': no locator available for file '/tmp/ipython-input-1307636700.py'

In [2]:
# 2) Data Ingestion & Preprocessing (Tabular + Graph) - WITH PROPER NORMALIZATION
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.impute import SimpleImputer
import time

TABULAR_PATH = '/content/eth_transactions_full_labeled.parquet'
NODE_FEATURES = ['sender_total_transactions','sender_total_sent','sender_total_received','sender_sophisticated_score','sender_hybrid_score']
LABEL_COL = 'fraud'

# ====== POLARS: 5-20x faster than pandas for parquet ======
print('📂 Loading data with Polars...')
t0 = time.time()

df_pl = pl.scan_parquet(TABULAR_PATH).select(
    NODE_FEATURES + [LABEL_COL, 'from_idx', 'to_idx']
).collect()

load_time = time.time() - t0
print(f'✅ Loaded {len(df_pl):,} rows in {load_time:.2f}s')

# Convert to numpy
X_raw = df_pl.select(NODE_FEATURES).to_numpy().astype(np.float64)  # float64 for preprocessing
y_np = df_pl[LABEL_COL].to_numpy().astype(np.float32)
from_idx_np = df_pl['from_idx'].to_numpy().astype(np.int64)
to_idx_np = df_pl['to_idx'].to_numpy().astype(np.int64)

# Free polars dataframe
del df_pl
gc.collect()

# ====== PREPROCESSING PIPELINE ======
print('\n🔧 Preprocessing features...')

# 1) Handle missing values (if any)
print('  1. Imputing missing values...')
imputer = SimpleImputer(strategy='median')  # Median is robust to outliers
X_imputed = imputer.fit_transform(X_raw)
n_missing = np.isnan(X_raw).sum()
if n_missing > 0:
    print(f'     Imputed {n_missing:,} missing values')
else:
    print('     No missing values found')

# 2) Analyze feature distributions
print('  2. Analyzing distributions...')
for i, feat in enumerate(NODE_FEATURES):
    col = X_imputed[:, i]
    skew = np.mean(((col - col.mean()) / col.std()) ** 3) if col.std() > 0 else 0
    print(f'     {feat}: min={col.min():.2f}, max={col.max():.2f}, skew={skew:.2f}')

# 3) Log transform for heavily skewed features (amounts, counts)
# Features with skew > 1 benefit from log transform
print('  3. Applying log1p transform to skewed features...')
SKEWED_FEATURES = ['sender_total_transactions', 'sender_total_sent', 'sender_total_received']
skewed_idx = [NODE_FEATURES.index(f) for f in SKEWED_FEATURES if f in NODE_FEATURES]

X_transformed = X_imputed.copy()
for idx in skewed_idx:
    # log1p handles zeros: log(1 + x)
    X_transformed[:, idx] = np.log1p(np.abs(X_transformed[:, idx]))
    print(f'     Applied log1p to {NODE_FEATURES[idx]}')

# 4) Choose scaler based on data characteristics
# - StandardScaler: assumes Gaussian, sensitive to outliers
# - RobustScaler: uses median/IQR, robust to outliers (RECOMMENDED for fraud)
# - QuantileTransformer: forces Gaussian, best for heavy tails
# - PowerTransformer: Box-Cox/Yeo-Johnson, handles skew

print('  4. Scaling features...')

# Option A: RobustScaler (recommended for fraud detection with outliers)
scaler = RobustScaler(quantile_range=(5, 95))  # Ignore extreme 5% on each side

# Option B: QuantileTransformer (if you want strict Gaussian)
# scaler = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=42)

# Option C: PowerTransformer (Yeo-Johnson handles negatives)
# scaler = PowerTransformer(method='yeo-johnson', standardize=True)

X_scaled = scaler.fit_transform(X_transformed)
print(f'     Scaler: {scaler.__class__.__name__}')

# 5) Clip extreme outliers (optional, for neural network stability)
print('  5. Clipping extreme values...')
CLIP_RANGE = 5  # Clip to ±5 std (after scaling)
X_clipped = np.clip(X_scaled, -CLIP_RANGE, CLIP_RANGE)
n_clipped = (np.abs(X_scaled) > CLIP_RANGE).sum()
print(f'     Clipped {n_clipped:,} extreme values (>{CLIP_RANGE} std)')

# 6) Verify preprocessing
print('\n📊 Preprocessed feature statistics:')
print(f"{'Feature':<35} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print('-' * 70)
for i, feat in enumerate(NODE_FEATURES):
    col = X_clipped[:, i]
    print(f'{feat:<35} {col.mean():>8.3f} {col.std():>8.3f} {col.min():>8.3f} {col.max():>8.3f}')

# ====== SAVE PREPROCESSORS FOR INFERENCE ======
preprocessors = {
    'imputer': imputer,
    'scaler': scaler,
    'skewed_idx': skewed_idx,
    'clip_range': CLIP_RANGE,
    'feature_names': NODE_FEATURES
}

# Move to GPU
X_tab = torch.from_numpy(X_clipped.astype(np.float32)).to(device)
y = torch.from_numpy(y_np).to(device)
from_idx = torch.from_numpy(from_idx_np)
to_idx = torch.from_numpy(to_idx_np)
edge_index = torch.stack([from_idx, to_idx], dim=0)

# Create PyG Data object
data = Data(x=X_tab, edge_index=edge_index, y=y)
print(f'\n✅ Graph: {data}')
print(f'   Fraud rate: {y_np.mean():.2%}')

free_mem()

📂 Loading data with Polars...


FileNotFoundError: No such file or directory (os error 2): /content/eth_transactions_full_labeled.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'select'
	[3] 'sink'


In [ ]:
# 3) β-VAE for Tabular Features (Unsupervised) with OneCycleLR + torch.compile
class BetaVAE(nn.Module):
    def __init__(self, in_dim, latent_dim=64, hidden=256, beta=4.0):
        super().__init__()
        self.beta = beta
        self.enc = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2), nn.LayerNorm(hidden//2), nn.GELU()
        )
        self.mu = nn.Linear(hidden//2, latent_dim)
        self.logvar = nn.Linear(hidden//2, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, hidden//2), nn.LayerNorm(hidden//2), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden//2, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, in_dim)
        )
    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    def decode(self, z):
        return self.dec(z)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

def vae_loss(recon, x, mu, logvar, beta=4.0):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss, kl

vae = BetaVAE(in_dim=X_tab.shape[1]).to(device)

# ====== torch.compile for 1.5-2x speedup ======
print('🔧 Compiling β-VAE with torch.compile...')
vae = compile_model(vae, mode="reduce-overhead")

opt = torch.optim.AdamW(vae.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = GradScaler()

# Train on NORMAL data only
normal_mask = (y == 0)
X_norm = X_tab[normal_mask]
loader = DataLoader(TensorDataset(X_norm), batch_size=1024, shuffle=True, pin_memory=True, num_workers=0)

# OneCycleLR for superconvergence
NUM_EPOCHS = 10
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=5e-3, epochs=NUM_EPOCHS, steps_per_epoch=len(loader),
    pct_start=0.3, div_factor=25, final_div_factor=1000
)

best_loss = float('inf')
patience_counter = 0
PATIENCE = 3

print(f'Training β-VAE on {len(X_norm):,} normal samples...')
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    vae.train(); total=0
    for (xb,) in loader:
        xb = xb.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast():
            recon, mu, logvar, z = vae(xb)
            loss, rloss, kl = vae_loss(recon, xb, mu, logvar, beta=4.0)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        scheduler.step()
        total += loss.item()*len(xb)
    avg_loss = total/len(loader.dataset)
    lr = scheduler.get_last_lr()[0]
    print(f'VAE Epoch {epoch+1}: loss={avg_loss:.4f}, lr={lr:.6f}')
    
    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(vae.state_dict(), '/content/vae_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

train_time = time.time() - t0
print(f'✅ VAE training completed in {train_time:.1f}s')

# Load best model
vae.load_state_dict(torch.load('/content/vae_best.pt'))
free_mem()

vae.eval()
with torch.no_grad():
    recon, mu, logvar, z = vae(X_tab)
    recon_err = ((recon - X_tab)**2).mean(dim=1)

X_tab_emb = torch.cat([z, recon_err.unsqueeze(1)], dim=1)
print(f'VAE embedding shape: {X_tab_emb.shape}')

In [ ]:
# 4) VGAE for Graph Structure (unsupervised) + torch.compile
from torch_geometric.nn import GCNConv, VGAE

class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 64)
        self.conv_mu = GCNConv(64, out_channels)
        self.conv_logvar = GCNConv(64, out_channels)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logvar(x, edge_index)

vgae = VGAE(Encoder(data.num_features + 1, 32)).to(device)

# ====== torch.compile for VGAE ======
# Note: GNN models may have limited torch.compile support, use default mode
print('🔧 Compiling VGAE...')
vgae = compile_model(vgae, mode="default")

opt_vgae = torch.optim.Adam(vgae.parameters(), lr=1e-3)
scaler = GradScaler()

# Augment node features with VAE recon error
node_x = torch.cat([X_tab, recon_err.unsqueeze(1)], dim=1)
data.x = node_x

print(f'Training VGAE on {data.num_nodes:,} nodes, {data.edge_index.shape[1]:,} edges...')
t0 = time.time()

loader_edges = DataLoader([data], batch_size=1)
for epoch in range(3):
    vgae.train()
    for batch in loader_edges:
        batch = batch.to(device)
        opt_vgae.zero_grad(set_to_none=True)
        with autocast():
            z = vgae.encode(batch.x, batch.edge_index)
            loss = vgae.recon_loss(z, batch.edge_index) + (1e-4)*vgae.kl_loss()
        scaler.scale(loss).backward()
        scaler.step(opt_vgae); scaler.update()
    print(f'VGAE epoch {epoch+1}: loss={loss.item():.4f}')

train_time = time.time() - t0
print(f'✅ VGAE training completed in {train_time:.1f}s')
free_mem()

vgae.eval()
with torch.no_grad():
    z_g = vgae.encode(node_x, data.edge_index)
    edge_recon = vgae.decoder.forward_all(z_g)

# Node embeddings with anomaly score
node_emb = torch.cat([z_g, edge_recon.mean(dim=1, keepdim=True)], dim=1)
print(f'Node embedding shape: {node_emb.shape}')

In [ ]:
# 5) GATv2 with Focal Loss + EMA + GraphSAINT + MC Dropout + torch.compile
from torch_geometric.nn import GATv2Conv
from torch_geometric.loader import NeighborLoader
try:
    from torch_geometric.loader import GraphSAINTRandomWalkSampler
    USE_SAINT = True
except ImportError:
    USE_SAINT = False
    print('GraphSAINT not available, using NeighborLoader')

# Focal Loss with Label Smoothing (SOTA for fraud detection)
class FocalLossSmooth(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
    
    def forward(self, logits, targets):
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets_smooth, reduction='none')
        pt = torch.exp(-bce)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = torch.where(targets >= 0.5, self.alpha, 1 - self.alpha)
        loss = alpha_weight * focal_weight * bce
        return loss.mean()

class GATModel(nn.Module):
    def __init__(self, in_dim, hidden=64, heads=4, dropout=0.2, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers
        
        # Multi-layer GATv2 with skip connections
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(GATv2Conv(in_dim, hidden, heads=heads, dropout=dropout, share_weights=True))
        self.bns.append(nn.BatchNorm1d(hidden * heads))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATv2Conv(hidden * heads, hidden, heads=heads, dropout=dropout, share_weights=True))
            self.bns.append(nn.BatchNorm1d(hidden * heads))
        
        self.convs.append(GATv2Conv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout))
        self.bns.append(nn.BatchNorm1d(hidden))
        
        # Skip connection projection
        self.skip_proj = nn.Linear(in_dim, hidden) if in_dim != hidden else nn.Identity()
        
        self.lin = nn.Linear(hidden, 1)
        self.attention_weights = None
    
    def forward(self, x, edge_index, return_attention=False):
        x_skip = self.skip_proj(x)
        
        for i, (conv, bn) in enumerate(zip(self.convs[:-1], self.bns[:-1])):
            x, attn = conv(x, edge_index, return_attention_weights=True)
            x = bn(x).relu()
            x = nn.functional.dropout(x, p=self.dropout, training=self.training)
        
        x, attn_last = self.convs[-1](x, edge_index, return_attention_weights=True)
        x = self.bns[-1](x).relu()
        
        # Skip connection
        x = x + x_skip
        
        if return_attention:
            self.attention_weights = attn_last
        
        return self.lin(x).squeeze(-1)

gat = GATModel(node_emb.shape[1], hidden=64, heads=4, num_layers=3).to(device)

# ====== torch.compile for GATv2 ======
print('🔧 Compiling GATv2 model...')
# Use default mode for GNN (reduce-overhead can have issues with dynamic shapes)
gat = compile_model(gat, mode="default")

opt_gat = torch.optim.AdamW(gat.parameters(), lr=1e-4, weight_decay=1e-4)
scaler = GradScaler()

# EMA for better generalization
ema_gat = EMA(gat, decay=0.999)

# Use GraphSAINT for scalable training if available, else NeighborLoader
if USE_SAINT:
    train_loader = GraphSAINTRandomWalkSampler(
        data, batch_size=4000, walk_length=3, num_steps=30, sample_coverage=100
    )
else:
    train_loader = NeighborLoader(
        data, num_neighbors=[25, 15, 10], batch_size=2048, shuffle=True, num_workers=0
    )

# OneCycleLR with longer training
NUM_EPOCHS_GAT = 20
total_steps = NUM_EPOCHS_GAT * len(train_loader)
scheduler_gat = torch.optim.lr_scheduler.OneCycleLR(
    opt_gat, max_lr=3e-3, total_steps=total_steps,
    pct_start=0.1, div_factor=25, final_div_factor=1000
)

# Focal Loss with label smoothing
focal_loss = FocalLossSmooth(alpha=0.75, gamma=2.0, smoothing=0.05)

best_gat_loss = float('inf')
patience_counter = 0
PATIENCE_GAT = 5

print(f'Training GATv2 with {"GraphSAINT" if USE_SAINT else "NeighborLoader"}...')
t0 = time.time()

for epoch in range(NUM_EPOCHS_GAT):
    gat.train()
    total = 0; n_samples = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        opt_gat.zero_grad(set_to_none=True)
        
        with autocast():
            if USE_SAINT:
                logits = gat(node_emb[batch.node_idx], batch.edge_index)
                labels = batch.y.float()
            else:
                logits = gat(node_emb[batch.n_id], batch.edge_index)
                labels = batch.y.float()
            loss = focal_loss(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(opt_gat)
        torch.nn.utils.clip_grad_norm_(gat.parameters(), 1.0)
        scaler.step(opt_gat)
        scaler.update()
        scheduler_gat.step()
        
        # Update EMA
        ema_gat.update(gat)
        
        total += loss.item() * len(labels)
        n_samples += len(labels)
    
    avg_loss = total / n_samples
    lr = scheduler_gat.get_last_lr()[0]
    print(f'GAT Epoch {epoch+1}: focal_loss={avg_loss:.4f}, lr={lr:.6f}')
    
    if avg_loss < best_gat_loss:
        best_gat_loss = avg_loss
        patience_counter = 0
        ema_gat.apply_shadow(gat)
        torch.save(gat.state_dict(), '/content/gat_best.pt')
        ema_gat.restore(gat)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE_GAT:
            print(f'GAT early stopping at epoch {epoch+1}')
            break

train_time = time.time() - t0
print(f'✅ GATv2 training completed in {train_time:.1f}s')

# Load best EMA model
gat.load_state_dict(torch.load('/content/gat_best.pt'))
free_mem()

# Inference with MC Dropout for uncertainty quantification
print('\n🔮 Computing predictions with uncertainty (MC Dropout)...')
gat_prob_mean, gat_prob_std = mc_dropout_predict(gat, node_emb, data.edge_index, n_samples=30)

# Also get deterministic prediction
gat.eval()
with torch.no_grad():
    gat_logits = gat(node_emb, data.edge_index, return_attention=True)
    gat_prob = torch.sigmoid(gat_logits)

print(f'GAT predictions: mean={gat_prob.mean():.4f}, std_uncertainty={gat_prob_std.mean():.4f}')
print(f'High uncertainty (>0.2): {(gat_prob_std > 0.2).sum().item():,} samples')

In [ ]:
# 6) XGBoost + LightGBM with K-Fold Ensemble + cuML acceleration
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

# ====== Use cuML if available for GPU-accelerated sklearn ======
if RAPIDS_AVAILABLE:
    from cuml.model_selection import train_test_split as gpu_train_test_split
    print('🚀 Using cuML GPU-accelerated train_test_split')
else:
    gpu_train_test_split = train_test_split
    print('ℹ️ Using sklearn CPU train_test_split')

# Prepare features
X_boost = torch.cat([X_tab, X_tab_emb.detach().cpu()], dim=1).cpu().numpy()
y_cpu = y.cpu().numpy()

# PROPER 3-WAY SPLIT: Train (60%) / Val (20%) / Test (20%)
t0 = time.time()
X_temp, X_test, y_temp, y_test = gpu_train_test_split(
    X_boost, y_cpu, test_size=0.20, stratify=y_cpu, random_state=42
)
X_train, X_val, y_train, y_val = gpu_train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)
split_time = time.time() - t0

print(f'Data split in {split_time:.2f}s: Train={len(y_train):,} | Val={len(y_val):,} | Test={len(y_test):,}')
print(f'Fraud rates: Train={y_train.mean():.2%} | Val={y_val.mean():.2%} | Test={y_test.mean():.2%}')

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# ====== K-FOLD ENSEMBLE FOR XGBOOST ======
print('\n🚀 Training XGBoost K-Fold Ensemble (GPU)...')
t0 = time.time()

N_FOLDS = 5
kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

xgb_params = {
    'objective': 'binary:logistic',
    'tree_method': 'gpu_hist',
    'eval_metric': 'aucpr',
    'max_depth': 6,
    'learning_rate': 0.03,
    'min_child_weight': 3,
    'gamma': 0.1,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.1,
    'reg_lambda': 1.5,
    'scale_pos_weight': scale_pos,
    'n_estimators': 2000,
    'gpu_id': 0,
    'n_jobs': -1
}

xgb_models = []
xgb_oof = np.zeros(len(X_temp))
xgb_test_preds = np.zeros((len(X_test), N_FOLDS))

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_temp, y_temp)):
    print(f'  Fold {fold+1}/{N_FOLDS}...', end=' ')
    
    X_tr, X_va = X_temp[train_idx], X_temp[val_idx]
    y_tr, y_va = y_temp[train_idx], y_temp[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    xgb_oof[val_idx] = model.predict_proba(X_va)[:, 1]
    xgb_test_preds[:, fold] = model.predict_proba(X_test)[:, 1]
    xgb_models.append(model)
    
    print(f'stopped at {model.best_iteration} trees')

xgb_test_prob = xgb_test_preds.mean(axis=1)
xgb_full_prob = np.zeros(len(X_boost))
xgb_full_prob[:len(X_temp)] = xgb_oof
xgb_full_prob[len(X_temp):] = xgb_test_prob

xgb_time = time.time() - t0
print(f'✅ XGBoost completed in {xgb_time:.1f}s | OOF ROC-AUC: {roc_auc_score(y_temp, xgb_oof):.4f}')

# ====== K-FOLD ENSEMBLE FOR LIGHTGBM ======
print('\n🚀 Training LightGBM K-Fold Ensemble (GPU)...')
t0 = time.time()

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'device': 'gpu',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'max_depth': 7,
    'learning_rate': 0.03,
    'min_child_samples': 20,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'is_unbalance': True,
    'n_estimators': 2000,
    'verbose': -1
}

lgb_models = []
lgb_oof = np.zeros(len(X_temp))
lgb_test_preds = np.zeros((len(X_test), N_FOLDS))

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_temp, y_temp)):
    print(f'  Fold {fold+1}/{N_FOLDS}...', end=' ')
    
    X_tr, X_va = X_temp[train_idx], X_temp[val_idx]
    y_tr, y_va = y_temp[train_idx], y_temp[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    lgb_oof[val_idx] = model.predict_proba(X_va)[:, 1]
    lgb_test_preds[:, fold] = model.predict_proba(X_test)[:, 1]
    lgb_models.append(model)
    
    print(f'stopped at {model.best_iteration_} trees')

lgb_test_prob = lgb_test_preds.mean(axis=1)
lgb_full_prob = np.zeros(len(X_boost))
lgb_full_prob[:len(X_temp)] = lgb_oof
lgb_full_prob[len(X_temp):] = lgb_test_prob

lgb_time = time.time() - t0
print(f'✅ LightGBM completed in {lgb_time:.1f}s | OOF ROC-AUC: {roc_auc_score(y_temp, lgb_oof):.4f}')

free_mem()

# ====== META-ENSEMBLE WITH STACKING (cuML if available) ======
print('\n🎯 Training Meta-Ensemble Stacking...')
t0 = time.time()

# Choose LogisticRegression implementation
if RAPIDS_AVAILABLE:
    from cuml.linear_model import LogisticRegression as GPULogisticRegression
    print('  Using cuML GPU LogisticRegression')
    
    # cuML doesn't have LogisticRegressionCV, so we use standard LR with good params
    meta = GPULogisticRegression(
        penalty='l2',
        C=1.0,
        class_weight='balanced',
        max_iter=2000,
        solver='qn'
    )
else:
    from sklearn.linear_model import LogisticRegressionCV
    meta = LogisticRegressionCV(
        Cs=np.logspace(-4, 2, 20),
        cv=5,
        scoring='neg_log_loss',
        class_weight='balanced',
        max_iter=3000,
        n_jobs=-1,
        solver='lbfgs'
    )

# Build meta features
recon_err_np = recon_err.detach().cpu().numpy()
edge_score_np = edge_recon.mean(dim=1).detach().cpu().numpy()
gat_prob_np = gat_prob.detach().cpu().numpy()
gat_uncertainty = gat_prob_std.detach().cpu().numpy()

# Meta features: include uncertainty as feature
meta_train_X = np.column_stack([
    recon_err_np[:len(X_temp)],
    edge_score_np[:len(X_temp)],
    gat_prob_np[:len(X_temp)],
    gat_uncertainty[:len(X_temp)],
    xgb_oof,
    lgb_oof
])

meta_test_X = np.column_stack([
    recon_err_np[len(X_temp):],
    edge_score_np[len(X_temp):],
    gat_prob_np[len(X_temp):],
    gat_uncertainty[len(X_temp):],
    xgb_test_prob,
    lgb_test_prob
])

meta_train_y = y_temp
meta_test_y = y_test

# Convert to cuDF if using RAPIDS
if RAPIDS_AVAILABLE:
    import cudf
    meta_train_X = cudf.DataFrame(meta_train_X)
    meta_train_y = cudf.Series(meta_train_y)

meta.fit(meta_train_X, meta_train_y)

# Convert back for predictions if needed
if RAPIDS_AVAILABLE:
    meta_train_X = meta_train_X.to_pandas().values
    meta_train_y = meta_train_y.to_pandas().values

meta_train_prob = meta.predict_proba(meta_train_X)[:, 1]
meta_test_prob = meta.predict_proba(meta_test_X)[:, 1]

# Handle cupy arrays if RAPIDS
if RAPIDS_AVAILABLE:
    meta_train_prob = np.array(meta_train_prob)
    meta_test_prob = np.array(meta_test_prob)

meta_time = time.time() - t0
print(f'✅ Meta-Ensemble completed in {meta_time:.1f}s')
print(f'  Train ROC-AUC: {roc_auc_score(y_temp, meta_train_prob):.4f}')
print(f'  Test ROC-AUC:  {roc_auc_score(y_test, meta_test_prob):.4f}')

# Full predictions for all data
meta_full_X = np.column_stack([
    recon_err_np,
    edge_score_np,
    gat_prob_np,
    gat_uncertainty,
    xgb_full_prob,
    lgb_full_prob
])
meta_prob = meta.predict_proba(meta_full_X)[:, 1]
if RAPIDS_AVAILABLE:
    meta_prob = np.array(meta_prob)

print('\n✅ K-Fold ensemble training complete!')
print(f'   Total training time: {xgb_time + lgb_time + meta_time:.1f}s')

In [ ]:
# 9) Comprehensive Evaluation on HOLDOUT TEST SET
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report, f1_score,
    precision_score, recall_score, accuracy_score, log_loss, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

print('='*70)
print('📊 COMPREHENSIVE MODEL EVALUATION (HOLDOUT TEST SET)')
print('='*70)

# Prepare test set predictions
test_probs = {
    'β-VAE (recon_err)': recon_err_np[len(X_temp):],
    'VGAE (edge_recon)': edge_score_np[len(X_temp):],
    'GATv2': gat_prob_np[len(X_temp):],
    'GAT Uncertainty': gat_uncertainty[len(X_temp):],
    'XGBoost (5-Fold)': xgb_test_prob,
    'LightGBM (5-Fold)': lgb_test_prob,
    'Meta-Ensemble': meta_test_prob
}

# Normalize unsupervised scores
scaler_recon = MinMaxScaler()
test_probs['β-VAE (recon_err)'] = scaler_recon.fit_transform(
    test_probs['β-VAE (recon_err)'].reshape(-1, 1)
).ravel()
scaler_edge = MinMaxScaler()
test_probs['VGAE (edge_recon)'] = scaler_edge.fit_transform(
    test_probs['VGAE (edge_recon)'].reshape(-1, 1)
).ravel()

# ====== TEST SET METRICS ======
print('\n📈 TEST SET METRICS (n={:,})'.format(len(y_test)))
print('-'*80)
print(f"{'Model':<22} {'ROC-AUC':>10} {'PR-AUC':>10} {'Brier':>10} {'LogLoss':>10}")
print('-'*80)

results = {}
for name, probs in test_probs.items():
    if name == 'GAT Uncertainty':
        continue  # Skip uncertainty, it's not a prediction
    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    brier = brier_score_loss(y_test, np.clip(probs, 0, 1))
    ll = log_loss(y_test, np.clip(probs, 1e-7, 1-1e-7))
    results[name] = {'ROC-AUC': roc, 'PR-AUC': pr, 'Brier': brier, 'LogLoss': ll}
    print(f"{name:<22} {roc:>10.4f} {pr:>10.4f} {brier:>10.4f} {ll:>10.4f}")

# Best model
best_model = max(results.items(), key=lambda x: x[1]['PR-AUC'])
print(f'\n🏆 Best Model (by PR-AUC): {best_model[0]} with PR-AUC={best_model[1]["PR-AUC"]:.4f}')

# ====== THRESHOLD OPTIMIZATION ======
print('\n🎯 THRESHOLD OPTIMIZATION (Meta-Ensemble on Test Set)')
precision, recall, thresholds_pr = precision_recall_curve(y_test, meta_test_prob)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds_pr[optimal_idx]

print(f'Optimal threshold (max F1): {optimal_threshold:.4f}')
print(f'  Precision: {precision[optimal_idx]:.4f}')
print(f'  Recall:    {recall[optimal_idx]:.4f}')
print(f'  F1:        {f1_scores[optimal_idx]:.4f}')

# Risk thresholds
THRESHOLDS = {
    'CRITICAL': 0.90,
    'HIGH': 0.75,
    'MEDIUM': 0.50,
    'LOW': 0.30
}

print('\n🚨 RISK LEVEL DISTRIBUTION (Test Set)')
for level, thresh in THRESHOLDS.items():
    if level == 'LOW':
        mask = (meta_test_prob >= thresh) & (meta_test_prob < THRESHOLDS['MEDIUM'])
    elif level == 'MEDIUM':
        mask = (meta_test_prob >= thresh) & (meta_test_prob < THRESHOLDS['HIGH'])
    elif level == 'HIGH':
        mask = (meta_test_prob >= thresh) & (meta_test_prob < THRESHOLDS['CRITICAL'])
    else:
        mask = meta_test_prob >= thresh
    count = mask.sum()
    fraud_count = (mask & (y_test == 1)).sum()
    precision_lvl = fraud_count / count if count > 0 else 0
    print(f'{level:>8}: {count:>6,} addresses ({fraud_count:>5,} true fraud, precision={precision_lvl:.1%})')

# Classification report
print('\n📋 CLASSIFICATION REPORT (Test Set, optimal threshold)')
preds_opt = (meta_test_prob >= optimal_threshold).astype(int)
print(classification_report(y_test, preds_opt, target_names=['Normal', 'Fraud'], digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, preds_opt)
tn, fp, fn, tp = cm.ravel()
print('🔢 CONFUSION MATRIX')
print(f'              Predicted')
print(f'              Normal    Fraud')
print(f'Actual Normal  {tn:>7,}  {fp:>7,}')
print(f'       Fraud   {fn:>7,}  {tp:>7,}')
print(f'\nSpecificity (TNR): {tn/(tn+fp):.4f}')
print(f'False Positive Rate: {fp/(tn+fp):.4f}')

# ====== UNCERTAINTY ANALYSIS ======
print('\n🔮 UNCERTAINTY ANALYSIS')
high_conf_mask = gat_uncertainty[len(X_temp):] < 0.1
low_conf_mask = gat_uncertainty[len(X_temp):] > 0.2
print(f'High confidence predictions (<0.1 std): {high_conf_mask.sum():,} ({high_conf_mask.mean():.1%})')
print(f'Low confidence predictions (>0.2 std):  {low_conf_mask.sum():,} ({low_conf_mask.mean():.1%})')

# ROC-AUC on high-confidence subset
if high_conf_mask.sum() > 100:
    roc_high_conf = roc_auc_score(y_test[high_conf_mask], meta_test_prob[high_conf_mask])
    print(f'ROC-AUC on high-confidence subset: {roc_high_conf:.4f}')

# ====== PLOTS ======
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. ROC Curves
ax = axes[0, 0]
for name, probs in test_probs.items():
    if name == 'GAT Uncertainty':
        continue
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name.split()[0]} ({roc:.3f})', linewidth=1.5)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (Test Set)')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

# 2. PR Curves
ax = axes[0, 1]
for name, probs in test_probs.items():
    if name == 'GAT Uncertainty':
        continue
    p, r, _ = precision_recall_curve(y_test, probs)
    pr = average_precision_score(y_test, probs)
    ax.plot(r, p, label=f'{name.split()[0]} ({pr:.3f})', linewidth=1.5)
ax.axhline(y_test.mean(), color='k', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (Test Set)')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

# 3. Calibration curves
ax = axes[0, 2]
for name in ['GATv2', 'XGBoost (5-Fold)', 'LightGBM (5-Fold)', 'Meta-Ensemble']:
    probs = test_probs[name]
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, marker='o', label=name.split()[0], linewidth=1.5)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves (Test Set)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 4. Score distributions with uncertainty
ax = axes[1, 0]
ax.hist(meta_test_prob[y_test == 0], bins=50, alpha=0.5, label='Normal', density=True, color='steelblue')
ax.hist(meta_test_prob[y_test == 1], bins=50, alpha=0.5, label='Fraud', density=True, color='coral')
ax.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal ({optimal_threshold:.2f})')
for level, thresh in THRESHOLDS.items():
    ax.axvline(thresh, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Density')
ax.set_title('Meta-Ensemble Score Distribution (Test Set)')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Uncertainty vs Prediction
ax = axes[1, 1]
scatter = ax.scatter(meta_test_prob, gat_uncertainty[len(X_temp):], 
                     c=y_test, cmap='coolwarm', alpha=0.3, s=5)
ax.set_xlabel('Fraud Probability')
ax.set_ylabel('Prediction Uncertainty (std)')
ax.set_title('Uncertainty vs Prediction')
ax.axhline(0.1, color='green', linestyle='--', alpha=0.7, label='Low uncertainty')
ax.axhline(0.2, color='orange', linestyle='--', alpha=0.7, label='High uncertainty')
ax.legend()
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('True Label')

# 6. Model comparison
ax = axes[1, 2]
model_names = [k for k in results.keys()]
roc_aucs = [results[m]['ROC-AUC'] for m in model_names]
pr_aucs = [results[m]['PR-AUC'] for m in model_names]
x = np.arange(len(model_names))
width = 0.35
bars1 = ax.bar(x - width/2, roc_aucs, width, label='ROC-AUC', color='steelblue')
bars2 = ax.bar(x + width/2, pr_aucs, width, label='PR-AUC', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([m.split()[0] for m in model_names], rotation=45, ha='right')
ax.set_ylabel('Score')
ax.set_title('Model Comparison (Test Set)')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/evaluation_test_set.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Evaluation complete! Plot saved to /content/evaluation_test_set.png')

In [ ]:
# 10) Advanced Visualizations: SHAP, Attention, Reconstruction Errors
import shap
import matplotlib.pyplot as plt
import seaborn as sns

print('='*60)
print('🎨 ADVANCED VISUALIZATIONS')
print('='*60)

# Feature names for interpretability
feature_names_tab = NODE_FEATURES
feature_names_boost = feature_names_tab + [f'vae_z{i}' for i in range(64)] + ['recon_err']

# ====== SHAP Analysis ======
print('\n🔍 Computing SHAP values (this may take a few minutes)...')

# Sample for SHAP (memory efficient)
sample_size = min(3000, len(X_boost))
sample_idx = np.random.choice(len(X_boost), size=sample_size, replace=False)
X_sample = X_boost[sample_idx]
y_sample = y_cpu[sample_idx]

# XGBoost SHAP
explainer_xgb = shap.TreeExplainer(xgb_best)
shap_values_xgb = explainer_xgb.shap_values(X_sample)

# LightGBM SHAP
explainer_lgb = shap.TreeExplainer(lgb_best)
shap_values_lgb = explainer_lgb.shap_values(X_sample)
if isinstance(shap_values_lgb, list):
    shap_values_lgb = shap_values_lgb[1]  # Class 1 (fraud)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. XGBoost SHAP Summary (beeswarm)
plt.sca(axes[0, 0])
shap.summary_plot(shap_values_xgb, X_sample, feature_names=feature_names_boost[:X_sample.shape[1]], 
                  max_display=15, show=False)
axes[0, 0].set_title('XGBoost SHAP Importance', fontsize=12)

# 2. LightGBM SHAP Summary
plt.sca(axes[0, 1])
shap.summary_plot(shap_values_lgb, X_sample, feature_names=feature_names_boost[:X_sample.shape[1]], 
                  max_display=15, show=False)
axes[0, 1].set_title('LightGBM SHAP Importance', fontsize=12)

# 3. Reconstruction Error Distribution by Class
ax = axes[1, 0]
recon_err_np = recon_err.detach().cpu().numpy()
fraud_recon = recon_err_np[y_cpu == 1]
normal_recon = recon_err_np[y_cpu == 0]

ax.hist(normal_recon, bins=100, alpha=0.6, label=f'Normal (n={len(normal_recon):,})', density=True, color='steelblue')
ax.hist(fraud_recon, bins=100, alpha=0.6, label=f'Fraud (n={len(fraud_recon):,})', density=True, color='coral')
ax.axvline(np.percentile(recon_err_np, 95), color='red', linestyle='--', label='95th percentile')
ax.axvline(np.percentile(recon_err_np, 99), color='darkred', linestyle=':', label='99th percentile')
ax.set_xlabel('Reconstruction Error')
ax.set_ylabel('Density')
ax.set_title('β-VAE Reconstruction Error Distribution')
ax.legend()
ax.set_xlim(0, np.percentile(recon_err_np, 99.5))
ax.grid(True, alpha=0.3)

# 4. Edge Reconstruction Score Distribution
ax = axes[1, 1]
edge_score = edge_recon.mean(dim=1).detach().cpu().numpy()
fraud_edge = edge_score[y_cpu == 1]
normal_edge = edge_score[y_cpu == 0]

ax.hist(normal_edge, bins=100, alpha=0.6, label='Normal', density=True, color='steelblue')
ax.hist(fraud_edge, bins=100, alpha=0.6, label='Fraud', density=True, color='coral')
ax.set_xlabel('Edge Reconstruction Score')
ax.set_ylabel('Density')
ax.set_title('VGAE Edge Reconstruction Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/shap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ====== GATv2 Attention Analysis ======
print('\n🔍 Analyzing GATv2 attention patterns...')

if gat.attention_weights is not None:
    attn1_edge_index, attn1_weights = gat.attention_weights[0]
    attn2_edge_index, attn2_weights = gat.attention_weights[1]
    
    # Aggregate attention per node
    attn1_np = attn1_weights.detach().cpu().numpy().mean(axis=1)  # Average over heads
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Attention weight distribution
    ax = axes[0]
    ax.hist(attn1_np, bins=100, alpha=0.7, color='purple')
    ax.set_xlabel('Attention Weight')
    ax.set_ylabel('Frequency')
    ax.set_title('GATv2 Layer 1 Attention Distribution')
    ax.grid(True, alpha=0.3)
    
    # High attention edges (potential important connections)
    high_attn_mask = attn1_np > np.percentile(attn1_np, 95)
    high_attn_edges = attn1_edge_index[:, high_attn_mask].detach().cpu().numpy()
    
    # Check if high-attention edges connect fraud nodes
    src_fraud = y_cpu[high_attn_edges[0]]
    dst_fraud = y_cpu[high_attn_edges[1]]
    both_fraud = (src_fraud == 1) & (dst_fraud == 1)
    any_fraud = (src_fraud == 1) | (dst_fraud == 1)
    
    ax = axes[1]
    labels = ['Both Normal', 'One Fraud', 'Both Fraud']
    sizes = [
        (~any_fraud).sum(),
        (any_fraud & ~both_fraud).sum(),
        both_fraud.sum()
    ]
    colors = ['lightgreen', 'orange', 'red']
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
    ax.set_title('High-Attention Edges by Fraud Status\n(top 5% attention)')
    
    # Attention vs node fraud probability
    ax = axes[2]
    node_attn = np.zeros(len(y_cpu))
    np.add.at(node_attn, attn1_edge_index[1].cpu().numpy(), attn1_np)
    
    ax.scatter(node_attn[y_cpu==0][:5000], gat_prob.cpu().numpy()[y_cpu==0][:5000], 
               alpha=0.1, s=1, label='Normal', c='steelblue')
    ax.scatter(node_attn[y_cpu==1][:5000], gat_prob.cpu().numpy()[y_cpu==1][:5000], 
               alpha=0.1, s=1, label='Fraud', c='coral')
    ax.set_xlabel('Aggregated Attention Received')
    ax.set_ylabel('GAT Fraud Probability')
    ax.set_title('Attention vs Fraud Prediction')
    ax.legend(markerscale=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/content/attention_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

# ====== SHAP Dependence Plots ======
print('\n📊 SHAP Dependence Plots for Top Features...')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Get top 3 features by mean absolute SHAP value
mean_shap = np.abs(shap_values_xgb).mean(axis=0)
top_features = np.argsort(mean_shap)[-3:][::-1]

for i, feat_idx in enumerate(top_features):
    ax = axes[i]
    shap.dependence_plot(feat_idx, shap_values_xgb, X_sample, 
                         feature_names=feature_names_boost[:X_sample.shape[1]], 
                         ax=ax, show=False)
    ax.set_title(f'Top {i+1}: {feature_names_boost[feat_idx]}')

plt.tight_layout()
plt.savefig('/content/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Visualizations saved!')
print('  - /content/shap_analysis.png')
print('  - /content/attention_analysis.png')
print('  - /content/shap_dependence.png')

In [ ]:
# 11) Model Checkpointing & Export
import joblib
import json
from datetime import datetime

print('='*60)
print('💾 MODEL CHECKPOINTING & EXPORT')
print('='*60)

# Create versioned output directory
VERSION = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'/content/amttp_models_{VERSION}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====== Save PyTorch Models ======
print('\n📦 Saving PyTorch models...')

# β-VAE
vae_checkpoint = {
    'model_state_dict': vae.state_dict(),
    'config': {
        'in_dim': X_tab.shape[1],
        'latent_dim': 64,
        'hidden': 256,
        'beta': 4.0
    }
}
torch.save(vae_checkpoint, f'{OUTPUT_DIR}/beta_vae.pt')
print(f'  ✓ β-VAE saved: {OUTPUT_DIR}/beta_vae.pt')

# VGAE
vgae_checkpoint = {
    'model_state_dict': vgae.state_dict(),
    'config': {
        'in_channels': data.num_features,
        'out_channels': 32
    }
}
torch.save(vgae_checkpoint, f'{OUTPUT_DIR}/vgae.pt')
print(f'  ✓ VGAE saved: {OUTPUT_DIR}/vgae.pt')

# GATv2
gat_checkpoint = {
    'model_state_dict': gat.state_dict(),
    'config': {
        'in_dim': node_emb.shape[1],
        'hidden': 64,
        'heads': 4,
        'dropout': 0.2
    }
}
torch.save(gat_checkpoint, f'{OUTPUT_DIR}/gatv2.pt')
print(f'  ✓ GATv2 saved: {OUTPUT_DIR}/gatv2.pt')

# ====== Save Scikit-learn / XGBoost / LightGBM Models ======
print('\n📦 Saving gradient boosting models...')

# XGBoost (native format for best compatibility)
xgb_best.save_model(f'{OUTPUT_DIR}/xgboost_fraud.ubj')
print(f'  ✓ XGBoost saved: {OUTPUT_DIR}/xgboost_fraud.ubj')

# LightGBM
lgb_best.booster_.save_model(f'{OUTPUT_DIR}/lightgbm_fraud.txt')
print(f'  ✓ LightGBM saved: {OUTPUT_DIR}/lightgbm_fraud.txt')

# Meta-ensemble (Logistic Regression)
joblib.dump(meta, f'{OUTPUT_DIR}/meta_ensemble.joblib')
print(f'  ✓ Meta-Ensemble saved: {OUTPUT_DIR}/meta_ensemble.joblib')

# Scalers (for VAE normalization during inference)
joblib.dump(scaler_recon, f'{OUTPUT_DIR}/scaler_recon.joblib')
joblib.dump(scaler_edge, f'{OUTPUT_DIR}/scaler_edge.joblib')

# ====== Save Preprocessors (for production inference) ======
print('\n🔧 Saving preprocessors...')
joblib.dump(preprocessors, f'{OUTPUT_DIR}/preprocessors.joblib')
print(f'  ✓ Preprocessors saved: {OUTPUT_DIR}/preprocessors.joblib')
print(f'    • RobustScaler: for outlier-resistant normalization')
print(f'    • StandardScaler: for graph node features')
print(f'    • MinMaxScaler: for final [0,1] range')
print(f'    • Log transform mask: {preprocessors["log_transform_mask"].sum()} features')
print(f'    • Skewness values: min={preprocessors["skewness"].min():.2f}, max={preprocessors["skewness"].max():.2f}')

# ====== Save Metadata & Config ======
print('\n📄 Saving metadata...')

metadata = {
    'version': VERSION,
    'training_date': datetime.now().isoformat(),
    'dataset': {
        'path': TABULAR_PATH,
        'n_samples': len(y_cpu),
        'n_features': X_tab.shape[1],
        'fraud_rate': float(y_cpu.mean()),
        'n_nodes': int(data.num_nodes) if hasattr(data, 'num_nodes') else len(y_cpu),
        'n_edges': int(data.edge_index.shape[1])
    },
    'preprocessing': {
        'log_transform_features': int(preprocessors["log_transform_mask"].sum()),
        'skewness_threshold': 1.0,
        'scalers': ['RobustScaler', 'MinMaxScaler'],
        'graph_scaler': 'StandardScaler'
    },
    'models': {
        'vae': {'type': 'BetaVAE', 'latent_dim': 64, 'beta': 4.0},
        'vgae': {'type': 'VGAE', 'out_channels': 32},
        'gat': {'type': 'GATv2', 'hidden': 64, 'heads': 4},
        'xgboost': {'best_params': rs_xgb.best_params_, 'n_estimators': xgb_best.best_iteration},
        'lightgbm': {'best_params': rs_lgb.best_params_, 'n_estimators': lgb_best.best_iteration_},
        'meta': {'type': 'LogisticRegressionCV', 'C': float(meta.C_[0])}
    },
    'thresholds': THRESHOLDS,
    'optimal_threshold': float(optimal_threshold),
    'performance': {
        'meta_roc_auc': float(results['Meta-Ensemble']['ROC-AUC']),
        'meta_pr_auc': float(results['Meta-Ensemble']['PR-AUC']),
        'xgb_roc_auc': float(results['XGBoost']['ROC-AUC']),
        'lgb_roc_auc': float(results['LightGBM']['ROC-AUC']),
        'gat_roc_auc': float(results['GATv2']['ROC-AUC'])
    }
}

with open(f'{OUTPUT_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'  ✓ Metadata saved: {OUTPUT_DIR}/metadata.json')

# ====== Save Feature Names ======
feature_config = {
    'tabular_features': NODE_FEATURES,
    'boost_features': feature_names_boost,
    'label_column': LABEL_COL
}
with open(f'{OUTPUT_DIR}/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
print(f'  ✓ Feature config saved: {OUTPUT_DIR}/feature_config.json')

# ====== Create Inference Script Template ======
inference_script = '''
"""
AMTTP Fraud Detection Inference Script
Generated: {version}

Includes full preprocessing pipeline for production inference.
"""
import torch
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import joblib
import json


def load_models(model_dir):
    """Load all models and preprocessors for inference."""
    # Load configs
    with open(f'{{model_dir}}/metadata.json') as f:
        metadata = json.load(f)
    with open(f'{{model_dir}}/feature_config.json') as f:
        feature_config = json.load(f)
    
    # Preprocessors (CRITICAL for production)
    preprocessors = joblib.load(f'{{model_dir}}/preprocessors.joblib')
    
    # XGBoost
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(f'{{model_dir}}/xgboost_fraud.ubj')
    
    # LightGBM  
    lgb_model = lgb.Booster(model_file=f'{{model_dir}}/lightgbm_fraud.txt')
    
    # Meta-ensemble
    meta_model = joblib.load(f'{{model_dir}}/meta_ensemble.joblib')
    
    return {{
        'xgb': xgb_model,
        'lgb': lgb_model,
        'meta': meta_model,
        'preprocessors': preprocessors,
        'metadata': metadata,
        'feature_config': feature_config
    }}


def preprocess_features(features, preprocessors):
    """
    Apply the same preprocessing used during training.
    
    Args:
        features: numpy array of shape (n_samples, n_features) - RAW features
        preprocessors: dict containing scalers and transforms
        
    Returns:
        Preprocessed features ready for model inference
    """
    X = features.copy().astype(np.float32)
    
    # 1. Handle missing values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
    # 2. Apply log transform to skewed features
    log_mask = preprocessors['log_transform_mask']
    X[:, log_mask] = np.log1p(np.clip(X[:, log_mask], 0, None))
    
    # 3. Apply RobustScaler (outlier-resistant)
    X = preprocessors['robust_scaler'].transform(X)
    
    # 4. Apply MinMaxScaler for final [0,1] range
    X = preprocessors['minmax_scaler'].transform(X)
    
    return X


def predict_fraud(raw_features, models, threshold=None):
    """
    Full inference pipeline with preprocessing.
    
    Args:
        raw_features: numpy array of shape (n_samples, n_features) - RAW unprocessed features
        models: dict returned by load_models()
        threshold: classification threshold (default: use optimal from training)
        
    Returns: 
        dict with fraud_prob, risk_levels, is_fraud, xgb_prob, lgb_prob
    """
    if threshold is None:
        threshold = models['metadata']['optimal_threshold']
    
    # Preprocess
    features = preprocess_features(raw_features, models['preprocessors'])
    
    # Predict with individual models
    xgb_prob = models['xgb'].predict_proba(features)[:, 1]
    lgb_prob = models['lgb'].predict(features)
    
    # Meta-ensemble (simplified - full version includes VAE and GATv2)
    meta_features = np.column_stack([xgb_prob, lgb_prob])
    fraud_prob = models['meta'].predict_proba(meta_features)[:, 1]
    
    # Risk levels
    risk_levels = np.where(fraud_prob >= 0.85, 'CRITICAL',
                  np.where(fraud_prob >= 0.65, 'HIGH',
                  np.where(fraud_prob >= 0.45, 'MEDIUM',
                  np.where(fraud_prob >= 0.25, 'LOW', 'MINIMAL'))))
    
    return {{
        'fraud_prob': fraud_prob,
        'risk_levels': risk_levels,
        'is_fraud': fraud_prob >= threshold,
        'threshold': threshold,
        'xgb_prob': xgb_prob,
        'lgb_prob': lgb_prob
    }}


if __name__ == '__main__':
    MODEL_DIR = '{output_dir}'
    models = load_models(MODEL_DIR)
    
    print(f"✅ Models loaded successfully!")
    print(f"   Performance: ROC-AUC={{models['metadata']['performance']['meta_roc_auc']:.4f}}")
    print(f"   Optimal threshold: {{models['metadata']['optimal_threshold']:.4f}}")
    print(f"   Features expected: {{len(models['feature_config']['tabular_features'])}}")
    
    # Example usage:
    # raw_data = np.random.randn(10, len(models['feature_config']['tabular_features']))
    # results = predict_fraud(raw_data, models)
    # print(f"Fraud probabilities: {{results['fraud_prob']}}")
    # print(f"Risk levels: {{results['risk_levels']}}")
'''.format(version=VERSION, output_dir=OUTPUT_DIR)

with open(f'{OUTPUT_DIR}/inference.py', 'w') as f:
    f.write(inference_script)
print(f'  ✓ Inference script saved: {OUTPUT_DIR}/inference.py')

# ====== Summary ======
print('\n' + '='*60)
print(f'✅ ALL MODELS SAVED TO: {OUTPUT_DIR}')
print('='*60)
print(f'''
Files created:
  📁 {OUTPUT_DIR}/
    ├── beta_vae.pt          (PyTorch β-VAE)
    ├── vgae.pt              (PyTorch VGAE) 
    ├── gatv2.pt             (PyTorch GATv2)
    ├── xgboost_fraud.ubj    (XGBoost native)
    ├── lightgbm_fraud.txt   (LightGBM native)
    ├── meta_ensemble.joblib (Sklearn LogisticRegression)
    ├── scaler_recon.joblib  (VAE reconstruction scaler)
    ├── scaler_edge.joblib   (VGAE edge scaler)
    ├── preprocessors.joblib (Full preprocessing pipeline) ⭐ NEW
    ├── metadata.json        (Training metadata & performance)
    ├── feature_config.json  (Feature names)
    └── inference.py         (Production inference script) ⭐ ENHANCED

Performance Summary:
  • Meta-Ensemble ROC-AUC: {results['Meta-Ensemble']['ROC-AUC']:.4f}
  • Meta-Ensemble PR-AUC:  {results['Meta-Ensemble']['PR-AUC']:.4f}
  • Optimal Threshold:     {optimal_threshold:.4f}
''')

# Zip for easy download
import shutil
shutil.make_archive(f'/content/amttp_models_{VERSION}', 'zip', OUTPUT_DIR)
print(f'📦 Download: /content/amttp_models_{VERSION}.zip')